In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_from_disk

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading Baseline Model: {model_id} on {device}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)

def run_baseline(review_text):
    prompt = f"<|system|>\nYou are a product analyst. Summarize the review in one short sentence and identify if it's biased.</s>\n<|user|>\nReview: {review_text}</s>\n<|assistant|>\n"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=60, 
            temperature=0.1, 
            do_sample=True,
            repetition_penalty=1.2
        )
    
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    result = decoded.split("<|assistant|>")[-1].strip()
    return result

def main():
    data_path = "augmentated_amazon_reviews_full/test"
    
    if not os.path.exists(data_path):
        print(f"Error: Dataset not found at {data_path}. Run your augmentation script first!")
        return

    print(f"Loading Test Subset from: {data_path}")
    test_ds = load_from_disk(data_path)

    print("\n" + "="*60)
    print("RUNNING BASELINE COMPARISON (Zero-shot / Prompt Engineered)")
    print("="*60)

    for i in range(5):
        sample = test_ds[i]
        review = sample['review_body']
        gold_summary = sample['summary']
        gold_bias = sample['bias_label']

        print(f"\n[Sample {i+1}]")
        print(f"Actual Review : {review[:200]}.")
        print(f"Target Summary: {gold_summary}")
        print(f"Target Bias   : {'Biased' if gold_bias == 1 else 'Non-Biased'}")
        
        print("-" * 30)
        
        baseline_output = run_baseline(review)
        print(f"Baseline LLM Output:\n{baseline_output}")
        print("-" * 60)

if __name__ == "__main__":
    main()

Loading Baseline Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0 on cuda...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 270.78it/s]
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loading Test Subset from: augmentated_amazon_reviews_full/test

RUNNING BASELINE COMPARISON (Zero-shot / Prompt Engineered)

[Sample 1]
Actual Review : Very disappointed in the quality of this product! For the amount I paid, not what it looks like in the pictures. And the box was squished on one whole side also....
Target Summary: Very disappointed in the quality of this product! For the amount I paid, not what it looks like in the pictures. And the box was squished on one whole side also.
Target Bias   : Non-Biased
------------------------------


[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Baseline LLM Output:
Bias: The reviewer is likely to be dissatisfied with the product due to its poor quality or shipping issues. However, they may have been provided an unboxing photo that does not accurately depict the condition of the item.
------------------------------------------------------------

[Sample 2]
Actual Review : The shirt is well designed and fits well. The color is really good...
Target Summary: Five Stars
Target Bias   : Non-Biased
------------------------------


...

Baseline LLM Output:
Based on the given text, the reviewer has rated the shirt as "well-designed" with an overall rating of 4 out of 5 stars. While the reviewer does mention that they found the color to be appealing, their overall assessment suggests that the sh
------------------------------------------------------------

[Sample 3]
Actual Review : The stickers are too small and the quality is not good....
Target Summary: Don't waste your time.
Target Bias   : Non-Biased
------------------------------


[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Baseline LLM Output:
Based on the given text, the reviewer has stated that they found the stickers to be too small and of poor quality. This suggests that the reviewer may have had an unfavorable experience with the product or service being reviewed. However, as there is no specific mention of any
------------------------------------------------------------

[Sample 4]
Actual Review : Material is cheaper than expected but ok for a crib skirt....
Target Summary: Not a great product.
Target Bias   : Non-Biased
------------------------------


[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Baseline LLM Output:
Based on the given material, the reviewer notes that the material used to make the crib skirt is not as expensive as they initially thought, which makes it an affordable option for those looking for a budget-friendly solution. However, the reviewer also mentions some issues with the
------------------------------------------------------------

[Sample 5]
Actual Review : The "leather" strap broke the 1st time I wore it, I just tied it back together for now. I will replace the strap, but the necklace is very pretty....
Target Summary: The "leather" strap broke the 1st time I wore it, I just tied it back together for now. I will replace the strap, but the necklace is very pretty.
Target Bias   : Non-Biased
------------------------------
Baseline LLM Output:
Based on the given text material, the reviewer states that they have worn the leather strap once before and ties it back together to keep it intact. However, they also mention replacing the strap as an option. Over

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_from_disk
import evaluate 
from tqdm import tqdm

# Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

print("Loading Baseline for Evaluation...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")

dataset = load_from_disk("augmentated_amazon_reviews_full/test")
test_subset = dataset.select(range(100)) 

predictions = []
references = []

print(f"Evaluating Baseline on {len(test_subset)} samples...")

for sample in tqdm(test_subset):
    review = sample['review_body']
    target = sample['summary']
    
    prompt = f"<|system|>\nSummarize this review in one short sentence.</s>\n<|user|>\nReview: {review}</s>\n<|assistant|>\n"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=40, temperature=0.1)
    
    gen_text = tokenizer.decode(outputs[0], skip_special_tokens=True).split("<|assistant|>")[-1].strip()
    
    predictions.append(gen_text)
    references.append(target)

rouge_results = rouge.compute(predictions=predictions, references=references)
bleu_results = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

print("\n" + "="*30)
print("BASELINE ROUGE SCORES")
print("="*30)
for key, value in rouge_results.items():
    print(f"{key}: {value:.4f}")
print("="*30)

print("\n" + "="*30)
print("BASELINE BLEU SCORES")
print("="*30)

for key, value in bleu_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("="*30)

c:\Users\manish\Downloads\Karanveer\GenAI-Project\augment\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Baseline for Evaluation...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:03<00:00, 65.83it/s]


Evaluating Baseline on 100 samples...


  0%|          | 0/100 [00:00<?, ?it/s][transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 100/100 [02:05<00:00,  1.26s/it]



BASELINE ROUGE SCORES
rouge1: 0.1904
rouge2: 0.0645
rougeL: 0.1553
rougeLsum: 0.1554

BASELINE BLEU SCORES
bleu: 0.0514
precisions: [0.18674105677715785, 0.058025110281642346, 0.03161222339304531, 0.02038587550054605]
brevity_penalty: 1.0000
length_ratio: 1.7176
translation_length: 3047
reference_length: 1774


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_from_disk
import evaluate
from tqdm import tqdm

# Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_path = "final_model" 

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

print("Loading Fine-Tuned Model for Evaluation...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(lora_path)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, lora_path)
model.eval()

dataset = load_from_disk("augmentated_amazon_reviews_full/test")

test_subset = dataset.select(range(100))

predictions = []
references = []

print(f"Evaluating Fine-Tuned Model on {len(test_subset)} samples...")

for sample in tqdm(test_subset):
    review = sample['review_body']
    target = sample['summary']

    prompt = (
        f"<|system|>\nSummarize and detect bias.</s>\n"
        f"<|user|>\n{review}</s>\n"
        f"<|assistant|>\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.1
        )

    gen_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    gen_text = gen_text.split("<|assistant|>")[-1].strip()
    gen_text = gen_text.split("| Bias")[0].replace("Summary:", "").strip()

    predictions.append(gen_text)
    references.append(target)

rouge_results = rouge.compute(predictions=predictions, references=references)

bleu_results = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

print("\n" + "="*30)
print("FINE-TUNED ROUGE SCORES")
print("="*30)
for key, value in rouge_results.items():
    print(f"{key}: {value:.4f}")

print("="*30)

print("\n" + "="*30)
print("FINE-TUNED BLEU SCORES")
print("="*30)

for key, value in bleu_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("="*30)

c:\Users\manish\Downloads\Karanveer\GenAI-Project\augment\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Fine-Tuned Model for Evaluation...


Loading weights: 100%|██████████| 201/201 [00:02<00:00, 88.63it/s] 


Evaluating Fine-Tuned Model on 100 samples...


  0%|          | 0/100 [00:00<?, ?it/s][transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 100/100 [02:52<00:00,  1.72s/it]


FINE-TUNED ROUGE SCORES
rouge1: 0.5092
rouge2: 0.4448
rougeL: 0.5039
rougeLsum: 0.5012

FINE-TUNED BLEU SCORES
bleu: 0.5492
precisions: [0.5905172413793104, 0.5444191343963554, 0.5344202898550725, 0.5295250320924262]
brevity_penalty: 1.0000
length_ratio: 1.0462
translation_length: 1856
reference_length: 1774
